<img src="img/mCIDaeNnb.png" alt="Logo CiDAEN" align="right">




<br><br><br>
<h2><font color="#00586D" size=4>Módulo 7</font></h2>



<h1><font color="#00586D" size=5>Capstone 7. Parte 2: Pronóstico de demanda eléctrica</font></h1>

<br><br><br>
<div style="text-align: right">
<font color="#00586D" size=3>Enrique González, Luis de la Ossa</font><br>
<font color="#00586D" size=3>Máster en Ciencia de Datos e Ingeniería de Datos en la Nube IV </font><br>
<font color="#00586D" size=3>Universidad de Castilla-La Mancha</font>

</div>

---

<a id="indice"></a>
<h2><font color="#00586D" size=5>Índice</font></h2>


* [1. Creación del modelo de pronóstico](#section1)
* [2. Ingesta y visualización del pronóstico](#section2)

In [1]:
import matplotlib.pyplot as plt
from influxdb_client import InfluxDBClient, Point, WritePrecision
from influxdb_client.client.write_api import SYNCHRONOUS
import pandas as pd
import numpy as np

# Optimiza los gráficos para pantalla retina
%config InlineBackend.figure_format = 'retina'
# Por defecto usamos el backend inline
%matplotlib inline

# Oculta warnings
import warnings
warnings.simplefilter('ignore')

# La libreta ocupa así el 95% de la pantalla
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:95% !important; }</style>"))

import logging
logger = logging.getLogger('cmdstanpy')
logger.addHandler(logging.NullHandler())
logger.propagate = False
logger.setLevel(logging.CRITICAL)

En la siguiente parte del capstone crearéis un modelo de pronóstico a partir de los datos ingestados en InfluxDB e añadiréis el pronóstico a un día vista de vuelta a InfluxDB de forma que en el sistema prototipado se pueda ver el pronóstico del día. Específicamente, para la creación del modelo utilizaréis los datos hasta el último día completo de datos y el pronóstico incluirá el dia siguiente (que podrá tener datos o no en nuestro sistema, pero que no los usaremos para entrenar). 

---

<a id="section1"></a>
## <font color="#00586D"> 1. Creación del modelo de pronóstico</font>
<br>

Para la creación del modelo de pronóstico, necesitaremos descargarnos los datos de nuestra instancia para poder tener datos con los que entrenar. 


--- 

### <font color="#004D7F"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#004D7F"></i> Tarea 1</font>
<br>

Implementa la siguiente función que devuelve un DataFrame con todos los datos disponibles de demanda eléctrica almacenados en tu instancia de InfluxDB. Solo nos interesa la serie de demanda real, por lo que tendréis que filtrar el Forecast de red eléctrica usando Flux. 

Necesitaréis añadir la información de autenticación para vuestra instancia de InfluxDB a continuación. 

_Pista_: La práctica de InfluxDB que hicimos en este módulo os resultará útil para poder formar la consulta. 

In [2]:
token = "mcidaen_token"
org = "mcidaen"
bucket = "capstone7"

client = InfluxDBClient(url="http://influxdb:8086", token=token)

In [4]:
def load_from_influxdb(client: InfluxDBClient, bucket: str, org: str) -> pd.DataFrame:
    """
    Devuelve un DataFrame con la serie temporal de RealDemand desde InfluxDB.
    """
    query = f'''
    from(bucket: "{bucket}")
      |> range(start: 0)
      |> filter(fn: (r) => r["_measurement"] == "demand")
      |> filter(fn: (r) => r["_field"] == "RealDemand")
      |> keep(columns: ["_time", "_value"])
    '''

    query_api = client.query_api()
    tables = query_api.query_data_frame(query=query, org=org)

    if isinstance(tables, list):  # En caso de múltiples tablas (puede pasar con algunos datos)
        df = pd.concat(tables, ignore_index=True)
    else:
        df = tables

    df_query = df.loc[:, ['_time', '_value']]
    df_query['datetime'] = pd.to_datetime(df_query['_time'], utc=True).dt.tz_localize(None)
    df_query['RealDemand'] = df_query['_value']
    df_query = df_query[['datetime', 'RealDemand']]
    df_query.index = df_query['datetime']
    return df_query['RealDemand'].sort_index()


In [5]:
df = load_from_influxdb(client, bucket, org)
df['2024'].sample(20, random_state=0).sort_index()

datetime
2024-01-14 05:00:00    21912.916667
2024-01-23 13:00:00    32208.833333
2024-02-22 22:00:00    28256.833333
2024-03-13 05:00:00    27400.250000
2024-04-09 02:00:00    21497.750000
2024-04-17 10:00:00    27627.500000
2024-06-03 02:00:00    21675.666667
2024-06-11 19:00:00    31106.000000
2024-06-15 05:00:00    22771.166667
2024-06-24 02:00:00    21709.416667
2024-08-16 11:00:00    28886.583333
2024-09-08 04:00:00    21697.666667
2024-09-21 17:00:00    26574.666667
2024-09-24 04:00:00    24955.250000
2024-09-30 08:00:00    29362.250000
2024-10-01 10:00:00    30193.250000
2024-10-07 20:00:00    30029.333333
2024-11-16 07:00:00    25059.250000
2024-12-14 12:00:00    29838.916667
2024-12-22 15:00:00    27064.000000
Name: RealDemand, dtype: float64

In [7]:
df_formatted = df['2024'].sample(20, random_state=0).sort_index().reset_index()
df_formatted.columns = ['datetime', 'RealDemand']
display(df_formatted)


,datetime,RealDemand
0,2024-01-14 05:00:00,21912.916667
1,2024-01-23 13:00:00,32208.833333
2,2024-02-22 22:00:00,28256.833333
3,2024-03-13 05:00:00,27400.250000
4,2024-04-09 02:00:00,21497.750000
5,2024-04-17 10:00:00,27627.500000
6,2024-06-03 02:00:00,21675.666667
7,2024-06-11 19:00:00,31106.000000
8,2024-06-15 05:00:00,22771.166667
9,2024-06-24 02:00:00,21709.416667


Resultado esperado (vuestro dataset puede variar):

| datetime            |   RealDemand |
|:--------------------|-------------:|
| 2024-01-02 07:00:00 |      28070.3 |
| 2024-01-07 06:00:00 |      20867.5 |
| 2024-01-13 11:00:00 |      29865.4 |
| 2024-01-13 18:00:00 |      31708.2 |
| 2024-01-13 22:00:00 |      27366.6 |
| 2024-01-16 04:00:00 |      23387.8 |
| 2024-01-19 00:00:00 |      24156.2 |
| 2024-01-19 19:00:00 |      34646.2 |
| 2024-01-22 09:00:00 |      33262.8 |
| 2024-01-22 17:00:00 |      33170.2 |
| 2024-01-25 16:00:00 |      30222.1 |
| 2024-01-25 21:00:00 |      30207.2 |
| 2024-01-28 15:00:00 |      22855.5 |
| 2024-01-28 18:00:00 |      27978   |
| 2024-01-29 09:00:00 |      31676   |
| 2024-01-30 07:00:00 |      32259.5 |
| 2024-01-31 02:00:00 |      22361.3 |
| 2024-01-31 16:00:00 |      30261.7 |
| 2024-02-01 06:00:00 |      30237.5 |
| 2024-02-01 21:00:00 |      30296.1 |


<div style="text-align: right"><font size=4> <i class="fa fa-check-square-o" aria-hidden="true" style="color:#00586D"></i></font></div>

--- 

Con estos datos, el objetivo es crear un modelo de forecasting que nos permita pronosticar a un día vista la demanda de energía horaria. Aunque podéis usar el modelo que estiméis (siempre que funcione con el resto de componentes) en este caso hemos optado por crear un modelo de Prophet. 

Para facilitar su integración con el resto de código y poder ejecutar el modelo para pronosticar varios días, crearemos una función que entrene el modelo y realice un pronóstico para un conjunto de test. Para ello crearemos un modelo capaz de pronosticar, dado un histórico, la demanda de energía horaria del día siguiente al histórico. 


--- 

### <font color="#004D7F"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#004D7F"></i> Tarea 2</font>
<br>

Implementa la siguiente función que, dada la serie temporal de demanda anterior transformada en darts.TimeSeries, _training_ y el número de pasos de pronóstico a predecir:
- Entrena un modelo en Darts  con los datos de _training_
- Pronostica las horas pasadas en _test_ 

Devuelve un TimeSeries con la serie pronósticada con el nombre de columna CP7Forecast

Opcional: Como trabajo opcional, puedes probar varios modelos y téecnicas de las expuestos durante el módulo para ver si puedes obtener un mejor resultado. Esto se tendrá en cuenta a la hora de evaluación del capstone. 

In [11]:
import datetime
from darts.models import Prophet
import darts
from darts import TimeSeries

def train_forecast(training: darts.TimeSeries, steps: int) -> darts.TimeSeries:

    model = Prophet()
    model.fit(training)
    forecast = model.predict(steps)
    forecast = forecast.with_columns_renamed(forecast.columns[0], "CP7Forecast")
    return forecast


In [9]:
training = darts.TimeSeries.from_dataframe(pd.DataFrame(df.loc[:datetime.datetime(2024, 2, 1),]))

In [10]:
result = train_forecast(training, 24)
result.pd_dataframe()

component,CP7Forecast
datetime,
2024-02-01 01:00:00,23878.005719
2024-02-01 02:00:00,22972.403732
2024-02-01 03:00:00,22651.954196
2024-02-01 04:00:00,23512.298712
2024-02-01 05:00:00,25743.387311
2024-02-01 06:00:00,28702.006133
2024-02-01 07:00:00,31239.705365
2024-02-01 08:00:00,32524.010106
2024-02-01 09:00:00,32597.765308


Resultado esperado (vuestro resultado puede variar):
    
| datetime            |  CP7Forecast |
|:--------------------|-------------:|
| 2024-02-01 01:00:00 |      25338.6 |
| 2024-02-01 02:00:00 |      24906.7 |
| 2024-02-01 03:00:00 |      25064.7 |
| 2024-02-01 04:00:00 |      26077.2 |
| 2024-02-01 05:00:00 |      27834.2 |
| 2024-02-01 06:00:00 |      29764.6 |
| 2024-02-01 07:00:00 |      31193   |
| 2024-02-01 08:00:00 |      31821.5 |
| 2024-02-01 09:00:00 |      31882.2 |
| 2024-02-01 10:00:00 |      31826.2 |
| 2024-02-01 11:00:00 |      31871.3 |
| 2024-02-01 12:00:00 |      31862.8 |
| 2024-02-01 13:00:00 |      31572.2 |
| 2024-02-01 14:00:00 |      31097.2 |
| 2024-02-01 15:00:00 |      30910.5 |
| 2024-02-01 16:00:00 |      31449.6 |
| 2024-02-01 17:00:00 |      32623.5 |
| 2024-02-01 18:00:00 |      33732.3 |
| 2024-02-01 19:00:00 |      33931.5 |
| 2024-02-01 20:00:00 |      32856.4 |
| 2024-02-01 21:00:00 |      30867.4 |
| 2024-02-01 22:00:00 |      28722.7 |
| 2024-02-01 23:00:00 |      27006.9 |
| 2024-02-02 00:00:00 |      25835   |

<div style="text-align: right"><font size=4> <i class="fa fa-check-square-o" aria-hidden="true" style="color:#00586D"></i></font></div>

--- 

Con esta función ya podemos evaluar nuestro modelo. En este caso, debemos recordar que debemos evaluar nuestro modelo de forma diaria, ya que nuestro objetivo es hacer pronósticos a un día vista. Para ello podemos valernos de la función anterior y recorrer diariamente los valores de un conjunto de test, añadiendo en el training los valores hasta la fecha. Solo usaremos los primeros días del mes de enero para probar el método. Veamos como hacerlo. 

In [12]:
test_dates = list(pd.date_range(start=datetime.datetime(2024, 1, 1), end=datetime.datetime(2024, 2, 1), freq='D'))

In [13]:
from darts.metrics import mape

mapes = []
for test_date in test_dates:
    train_df = pd.DataFrame(df.loc[:test_date-datetime.timedelta(minutes=1),])
    training = darts.TimeSeries.from_dataframe(train_df)
    test_df = pd.DataFrame(df[test_date:test_date+datetime.timedelta(days=1, minutes=-1)])
    test = darts.TimeSeries.from_dataframe(test_df)
    test_steps = len(test_df)
    result = train_forecast(training, test_steps)
    r = mape(test, result)
    mapes.append(r)

In [14]:
np.mean(mapes)

9.805162208019903

Con este modelo ya podemos añadir la información de pronóstico como un nuevo campo en nuestra serie en InfluxDB. Para ello seguiremos la siguiente estrategia:
- Nos centraremos solo en el último més de datos disponible. Para el último día, que posiblemente no estará completo, deberemos añadir las horas que falten. 
- Generaremos un pronóstico para cada día usando el modelo que creamos anteriormente. 
- Subiremos este pronóstico directamente a InfluxDB

Implementaremos este proceso por partes a continuación. 


--- 

### <font color="#004D7F"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#004D7F"></i> Tarea 3 </font>
<br>

Implementa la siguiente función que, dada una fecha, devuelve una lista con los timestamps ordenados de sus horas. 

In [22]:
def get_hours(date: datetime) -> list:
    """
    Devuelve una lista con los timestamps horarios de la fecha dada.
    """
    return list(pd.date_range(start=date, periods=24, freq='H'))


In [23]:
get_hours(datetime(2023, 2, 28))

[Timestamp('2023-02-28 00:00:00', freq='H'),
 Timestamp('2023-02-28 01:00:00', freq='H'),
 Timestamp('2023-02-28 02:00:00', freq='H'),
 Timestamp('2023-02-28 03:00:00', freq='H'),
 Timestamp('2023-02-28 04:00:00', freq='H'),
 Timestamp('2023-02-28 05:00:00', freq='H'),
 Timestamp('2023-02-28 06:00:00', freq='H'),
 Timestamp('2023-02-28 07:00:00', freq='H'),
 Timestamp('2023-02-28 08:00:00', freq='H'),
 Timestamp('2023-02-28 09:00:00', freq='H'),
 Timestamp('2023-02-28 10:00:00', freq='H'),
 Timestamp('2023-02-28 11:00:00', freq='H'),
 Timestamp('2023-02-28 12:00:00', freq='H'),
 Timestamp('2023-02-28 13:00:00', freq='H'),
 Timestamp('2023-02-28 14:00:00', freq='H'),
 Timestamp('2023-02-28 15:00:00', freq='H'),
 Timestamp('2023-02-28 16:00:00', freq='H'),
 Timestamp('2023-02-28 17:00:00', freq='H'),
 Timestamp('2023-02-28 18:00:00', freq='H'),
 Timestamp('2023-02-28 19:00:00', freq='H'),
 Timestamp('2023-02-28 20:00:00', freq='H'),
 Timestamp('2023-02-28 21:00:00', freq='H'),
 Timestamp

Resutado esperado:
```
[Timestamp('2023-02-28 00:00:00', freq='H'),
 Timestamp('2023-02-28 01:00:00', freq='H'),
 Timestamp('2023-02-28 02:00:00', freq='H'),
 Timestamp('2023-02-28 03:00:00', freq='H'),
 Timestamp('2023-02-28 04:00:00', freq='H'),
 Timestamp('2023-02-28 05:00:00', freq='H'),
 Timestamp('2023-02-28 06:00:00', freq='H'),
 Timestamp('2023-02-28 07:00:00', freq='H'),
 Timestamp('2023-02-28 08:00:00', freq='H'),
 Timestamp('2023-02-28 09:00:00', freq='H'),
 Timestamp('2023-02-28 10:00:00', freq='H'),
 Timestamp('2023-02-28 11:00:00', freq='H'),
 Timestamp('2023-02-28 12:00:00', freq='H'),
 Timestamp('2023-02-28 13:00:00', freq='H'),
 Timestamp('2023-02-28 14:00:00', freq='H'),
 Timestamp('2023-02-28 15:00:00', freq='H'),
 Timestamp('2023-02-28 16:00:00', freq='H'),
 Timestamp('2023-02-28 17:00:00', freq='H'),
 Timestamp('2023-02-28 18:00:00', freq='H'),
 Timestamp('2023-02-28 19:00:00', freq='H'),
 Timestamp('2023-02-28 20:00:00', freq='H'),
 Timestamp('2023-02-28 21:00:00', freq='H'),
 Timestamp('2023-02-28 22:00:00', freq='H'),
 Timestamp('2023-02-28 23:00:00', freq='H')]
```

<div style="text-align: right"><font size=4> <i class="fa fa-check-square-o" aria-hidden="true" style="color:#00586D"></i></font></div>

--- 

Para ingestar el pronóstico en InfluxDB, podemos usar la siguiente función, similar a la función que utilizásteis en la primera práctica y que básicamente toma el pronóstico del modelo de Prophet y lo ingesta usando la API de escritura de InfluxDB. El pronóstico se ingestará en la misma medida que la demanda, pero con el campo `CP7Forecast`.

In [24]:
def save_to_influxdb(df: pd.DataFrame, client: InfluxDBClient, bucket: str, org: str) -> pd.DataFrame:
    """
    Escribe el pronóstico en InfluxDB
    """
    df = df.reset_index()
    df['time'] = df['datetime']
    to_write = df[['time', 'CP7Forecast']]
    to_write = to_write.set_index('time')[['CP7Forecast']]
    write_api = client.write_api(write_options=SYNCHRONOUS)
    write_api.write(bucket, org, record=to_write, data_frame_measurement_name='demand')

Con estas funciones ya tenemos todo lo necesario para poder escribir en InfluxDB los datos de pronóstico. Para ello, partiendo de un día inicial desde el que empezaremos a generar pronósticos, usaremos una estrategia similar al método de evaluación anterior, simplemente teniendo en cuenta que generaremos nosotros las horas en vez de usar las que aparecen en un conjunto de test. A continuación mostramos como ingestar los resultados a partir de un día determinado. Podéis ejecutar el código a continuación para añadir el pronóstico a vuestra instancia de InfluxDB.

In [29]:
begin = datetime(2025, 5,1)
dates_to_ingest = pd.date_range(start=begin, end=datetime.now().date(), freq='D')
dates_to_ingest

DatetimeIndex(['2025-05-01', '2025-05-02', '2025-05-03', '2025-05-04',
               '2025-05-05', '2025-05-06', '2025-05-07', '2025-05-08',
               '2025-05-09', '2025-05-10', '2025-05-11', '2025-05-12',
               '2025-05-13', '2025-05-14', '2025-05-15', '2025-05-16',
               '2025-05-17', '2025-05-18', '2025-05-19', '2025-05-20',
               '2025-05-21'],
              dtype='datetime64[ns]', freq='D')

In [30]:
for dt in dates_to_ingest:
    hours = len(get_hours(dt))
    print(f"Ingesting for date: {dt}")
    train_df = pd.DataFrame(df[:dt-timedelta(minutes=1)])
    training = darts.TimeSeries.from_dataframe(train_df, freq='H', fill_missing_dates=True)
    result = train_forecast(training, hours).pd_dataframe()
    save_to_influxdb(result, client, bucket, org)

Ingesting for date: 2025-05-01 00:00:00
Ingesting for date: 2025-05-02 00:00:00
Ingesting for date: 2025-05-03 00:00:00
Ingesting for date: 2025-05-04 00:00:00
Ingesting for date: 2025-05-05 00:00:00
Ingesting for date: 2025-05-06 00:00:00
Ingesting for date: 2025-05-07 00:00:00
Ingesting for date: 2025-05-08 00:00:00
Ingesting for date: 2025-05-09 00:00:00
Ingesting for date: 2025-05-10 00:00:00
Ingesting for date: 2025-05-11 00:00:00
Ingesting for date: 2025-05-12 00:00:00
Ingesting for date: 2025-05-13 00:00:00
Ingesting for date: 2025-05-14 00:00:00
Ingesting for date: 2025-05-15 00:00:00
Ingesting for date: 2025-05-16 00:00:00
Ingesting for date: 2025-05-17 00:00:00
Ingesting for date: 2025-05-18 00:00:00
Ingesting for date: 2025-05-19 00:00:00
Ingesting for date: 2025-05-20 00:00:00
Ingesting for date: 2025-05-21 00:00:00


Cuando acabe el proceso anterior ya deberíais tener los resultados en InfluxDB. Solo quedaría adaptar el dashboard anterior para visualizar también los datos de pronóstico. 


--- 

### <font color="#004D7F"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#004D7F"></i> Tarea 4</font>
<br>

Añade a la celda tipo graph del dashboard que creaste en la primera parte del capstone la serie predicha por el modelo que hemos entrenado.

Para visualizar el pronóstico a futuro, necesitarás modificar el filtro de tiempo para incluir el día en curso. 

**Añade una captura de tu dashboard a continuación (incluyéndo el dashboard con la entrega)**. 


Resultado esperado (aprox):

![tarea4](img/p2tarea4.png)

Mi Resultado

![tarea4](img_luis/dashboard_2.png)

<div style="text-align: right"><font size=4> <i class="fa fa-check-square-o" aria-hidden="true" style="color:#00586D"></i></font></div>

--- 